In [3]:
import time
import torch
import torch.nn.functional as F
import torchaudio
from torchaudio import transforms
import numpy as np
import bc_resnet_model

In [5]:
# ================================
# 설정
# ================================
EPS         = 1e-9
SAMPLE_RATE = 16000
TARGET_LEN  = 16000  # 1초
SCALE       = 2
N_CLASS     = 3
CLASSES     = ['silence', 'unknown', 'wakeword']
THRESHOLD   = 0.8

# ================================
# 모델 로드
# ================================
device = torch.device('cpu')
model  = bc_resnet_model.BcResNetModel(
    n_class=N_CLASS,
    scale=SCALE,
    use_subspectral=True
)

import torch.nn as nn
model.head_conv = nn.Conv2d(32 * SCALE, N_CLASS, kernel_size=1)
model.load_state_dict(torch.load('./best_horiya2_3.pt', map_location='cpu'))
model.eval()
print("모델 로드 완료!")

# ================================
# 전처리 (re9ulus 방식)
# ================================
to_mel = transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=1024,
    f_max=8000,
    n_mels=40
)

def preprocess(waveform, sr):
    # 스테레오 → 모노
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # 리샘플링
    if sr != SAMPLE_RATE:
        waveform = transforms.Resample(sr, SAMPLE_RATE)(waveform)

    # 패딩/자르기 (1초)
    length = waveform.shape[1]
    if length > TARGET_LEN:
        waveform = waveform[:, :TARGET_LEN]
    elif length < TARGET_LEN:
        waveform = F.pad(waveform, (0, TARGET_LEN - length))

    # 로그 멜 스펙트로그램
    log_mel = (to_mel(waveform) + EPS).log2()
    return log_mel.unsqueeze(0)  # (1, 1, 40, T)

RuntimeError: Error(s) in loading state_dict for BcResNetModel:
	size mismatch for head_conv.weight: copying a param with shape torch.Size([2, 64, 1, 1]) from checkpoint, the shape in current model is torch.Size([3, 64, 1, 1]).
	size mismatch for head_conv.bias: copying a param with shape torch.Size([2]) from checkpoint, the shape in current model is torch.Size([3]).

In [ ]:
# ================================
# 테스트
# ================================
wav_path = '/home/isol/work/bc_resnet_re9ulus/test_audio/detection_1772772229.wav'
# wav_path = '/home/user/work/bc_resnet/raw/wakeword_speech_noisy2/20251203_171623_MIC0_noisy.wav'

waveform, sr = torchaudio.load(wav_path)
spec = preprocess(waveform, sr)

with torch.no_grad():
    start = time.time()
    output = model(spec)
    end = time.time()
    probs  = F.softmax(output.squeeze(), dim=0)

inference_ms = (end - start) * 1000

print("="*35)
for i, cls in enumerate(CLASSES):
    bar    = "█" * int(probs[i].item() * 20)
    marker = " ← 감지!" if (cls == 'wakeword' and probs[i].item() >= THRESHOLD) else ""
    print(f"{cls:10s}: {probs[i].item()*100:5.1f}% {bar}{marker}")
print("="*35)
print(f"추론 시간: {inference_ms:.2f}ms")
print(f"30ms 실시간 가능 여부: {'✅ 가능' if inference_ms < 30 else '❌ 느림, 스텝 늘려야 함'}")

silence   :   0.0% 
unknown   :   3.3% 
wakeword  :  96.7% ███████████████████ ← 감지!
추론 시간: 5.38ms
30ms 실시간 가능 여부: ✅ 가능
